# GR00T Experiments Pipeline

Basic notebook for loading a small GR00T demo dataset, optionally running GR00T policy inference, saving execution traces, and checking STL properties with the RTAMT Python library.

The model load is intentionally behind `RUN_MODEL`: GR00T checkpoints can be large and may download weights from Hugging Face. With `RUN_MODEL = False`, the notebook still loads an episode and builds a replay trace from dataset actions so the trace and monitor workflow can be exercised quickly.

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd


os.environ["CUDA_VISIBLE_DEVICES"] = "2"
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "external" / "Isaac-GR00T").exists():
    REPO_ROOT = next(p for p in Path.cwd().parents if (p / "external" / "Isaac-GR00T").exists())

GROOT_ROOT = REPO_ROOT / "external" / "Isaac-GR00T"
SRC_ROOT = REPO_ROOT / "src"
for path in (GROOT_ROOT, SRC_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

OUTPUT_DIR = REPO_ROOT / "examples" / "outputs" / "groot_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo: {REPO_ROOT}")
print(f"outputs: {OUTPUT_DIR}")

repo: /home/ktit/repos/rfm-validator
outputs: /home/ktit/repos/rfm-validator/examples/outputs/groot_pipeline


## Configuration

The DROID sample works with the base GR00T model and DROID embodiment tag.

`MODEL_PATH` accepts either:
- a Hugging Face model ID (for example `nvidia/GR00T-N1.7-3B`)
- a local checkpoint directory path

For other datasets, update `DATASET_PATH`, `MODEL_PATH`, and `EMBODIMENT_TAG` together so they stay compatible.

In [2]:
RUN_MODEL = False
MODEL_PATH = "nvidia/GR00T-N1.7-3B"  # Hugging Face repo ID (valid)
DATASET_PATH = GROOT_ROOT / "demo_data" / "droid_sample"
EMBODIMENT_TAG = "OXE_DROID_RELATIVE_EEF_RELATIVE_JOINT"

# Optional local checkpoint example:
# MODEL_PATH = str(REPO_ROOT / "checkpoints" / "gr00t" / "checkpoint-2000")

EPISODE_ID = 0
MAX_STEPS = 64
ACTION_HORIZON = 16
DT = 0.05

# Other useful local samples:
# DATASET_PATH = GROOT_ROOT / "demo_data" / "cube_to_bowl_5"          # requires a compatible finetuned checkpoint
# DATASET_PATH = GROOT_ROOT / "demo_data" / "simplerenv_bridge_sample" # use SIMPLER_ENV_WIDOWX checkpoint/tag
# DATASET_PATH = GROOT_ROOT / "demo_data" / "simplerenv_fractal_sample" # use SIMPLER_ENV_GOOGLE checkpoint/tag

print(DATASET_PATH)

/home/ktit/repos/rfm-validator/external/Isaac-GR00T/demo_data/droid_sample


## Load A Toy GR00T Dataset

In [3]:
def read_json(path: Path) -> Any:
    with path.open() as f:
        return json.load(f)

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]

info = read_json(DATASET_PATH / "meta" / "info.json")
modality = read_json(DATASET_PATH / "meta" / "modality.json")
episodes = read_jsonl(DATASET_PATH / "meta" / "episodes.jsonl")
tasks = read_jsonl(DATASET_PATH / "meta" / "tasks.jsonl")

episode_path = DATASET_PATH / "data" / "chunk-000" / f"episode_{EPISODE_ID:06d}.parquet"
episode_df = pd.read_parquet(episode_path)
episode_df = episode_df.head(MAX_STEPS).copy()

task_lookup = {row.get("task_index", row.get("index")): row for row in tasks}
task_index = int(episode_df["task_index"].iloc[0]) if "task_index" in episode_df else None
task_text = task_lookup.get(task_index, {}).get("task", "")

print(f"dataset episodes: {len(episodes)}")
print(f"loaded episode: {episode_path.name}, rows={len(episode_df)}")
print(f"task: {task_text!r}")
episode_df.head()

dataset episodes: 3
loaded episode: episode_000000.parquet, rows=64
task: ''


,is_first,is_last,is_terminal,language_instruction,language_instruction_2,language_instruction_3,observation.state.gripper_position,observation.state.cartesian_position,observation.state.joint_position,observation.state,...,camera_extrinsics.exterior_1_left,camera_extrinsics.exterior_2_left,is_episode_successful,timestamp,frame_index,episode_index,index,task_index,observation.state.eef_9d,action.eef_9d
0,True,False,False,,,,0.0,"[0.41833704710006714, -0.1657044142484665, 0.5...","[-0.224760040640831, -0.4210602343082428, -0.1...","[0.41833704710006714, -0.1657044142484665, 0.5...",...,"[0.3311587870121002, 0.7093092799186707, 0.495...","[-0.061356283724308014, -0.599903404712677, 0....",True,0.000000,0,0,0,0,"[0.41833705, -0.16570441, 0.51653355, -0.24825...","[0.41801384, -0.16655575, 0.51651996, -0.25011..."
1,False,False,False,,,,0.0,"[0.4179970622062683, -0.16660013794898987, 0.5...","[-0.22599239647388458, -0.4210450351238251, -0...","[0.4179970622062683, -0.16660013794898987, 0.5...",...,"[0.3311587870121002, 0.7093092799186707, 0.495...","[-0.061356283724308014, -0.599903404712677, 0....",True,0.066667,1,0,1,0,"[0.41799706, -0.16660014, 0.5165134, -0.250214...","[0.41768798, -0.16740127, 0.51649386, -0.25192..."
2,False,False,False,,,,0.0,"[0.41767990589141846, -0.16742141544818878, 0....","[-0.2264527976512909, -0.42108213901519775, -0...","[0.41767990589141846, -0.16742141544818878, 0....",...,"[0.3311587870121002, 0.7093092799186707, 0.495...","[-0.061356283724308014, -0.599903404712677, 0....",True,0.133333,2,0,2,0,"[0.4176799, -0.16742142, 0.5164939, -0.2519716...","[0.4174855, -0.16794257, 0.5164566, -0.2530664..."
3,False,False,False,,,,0.0,"[0.41747966408729553, -0.16795647144317627, 0....","[-0.22645342350006104, -0.4210798144340515, -0...","[0.41747966408729553, -0.16795647144317627, 0....",...,"[0.3311587870121002, 0.7093092799186707, 0.495...","[-0.061356283724308014, -0.599903404712677, 0....",True,0.200000,3,0,3,0,"[0.41747966, -0.16795647, 0.5164576, -0.253098...","[0.41741836, -0.16813047, 0.5164369, -0.253462..."
4,False,False,False,,,,0.0,"[0.4174184203147888, -0.16813358664512634, 0.5...","[-0.22645288705825806, -0.4210563004016876, -0...","[0.4174184203147888, -0.16813358664512634, 0.5...",...,"[0.3311587870121002, 0.7093092799186707, 0.495...","[-0.061356283724308014, -0.599903404712677, 0....",True,0.266667,4,0,4,0,"[0.41741842, -0.16813359, 0.51643777, -0.25346...","[0.417425, -0.1681063, 0.51644653, -0.25341022..."


In [4]:
state_columns = [c for c in episode_df.columns if c.startswith("state.")]
action_columns = [c for c in episode_df.columns if c.startswith("action.")]
video_columns = [c for c in episode_df.columns if c.startswith("observation.images.") or c.startswith("video.")]

print("state columns:", state_columns)
print("action columns:", action_columns)
print("video columns:", video_columns[:5])
print("metadata keys:", sorted(info.keys()))

state columns: []
action columns: ['action.gripper_position', 'action.gripper_velocity', 'action.cartesian_position', 'action.cartesian_velocity', 'action.joint_position', 'action.joint_velocity', 'action.original', 'action.eef_9d']
video columns: []
metadata keys: ['chunks_size', 'codebase_version', 'data_path', 'features', 'fps', 'robot_type', 'splits', 'total_episodes', 'total_frames', 'video_path']


## Optional: Load GR00T And Compute Action Executions

Set `RUN_MODEL = True` above to load `Gr00tPolicy` and compute predicted actions at chunk boundaries. The fallback replay path records dataset actions as executions, which keeps the trace and STL sections runnable on lightweight machines.

In [8]:
def as_array(value: Any) -> np.ndarray:
    arr = np.asarray(value)
    if arr.dtype == object and arr.shape == ():
        arr = np.asarray(arr.item())
    return arr

def flatten_row_arrays(row: pd.Series, columns: list[str]) -> np.ndarray:
    values = [np.atleast_1d(as_array(row[col])).astype(float).ravel() for col in columns]
    return np.concatenate(values) if values else np.array([], dtype=float)

def to_jsonable(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, dict):
        return {k: to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value

executions: list[dict[str, Any]] = []

if RUN_MODEL and sys.version_info[:2] != (3, 10):
    raise RuntimeError("GR00T model loading currently requires a Python 3.10 kernel. Recreate the uv env with `uv sync --python 3.10` and select that kernel.")

if RUN_MODEL:
    import torch
    from copy import deepcopy
    from gr00t.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader
    from gr00t.data.dataset.sharded_single_step_dataset import extract_step_data
    from gr00t.data.embodiment_tags import EmbodimentTag
    from gr00t.policy.gr00t_policy import Gr00tPolicy

    device = "cuda" if torch.cuda.is_available() else "cpu"
    policy = Gr00tPolicy(
        model_path=MODEL_PATH,
        embodiment_tag=EmbodimentTag.resolve(EMBODIMENT_TAG),
        device=device
       
    )
    modality_config = policy.get_modality_config()
    loader = LeRobotEpisodeLoader(dataset_path=str(DATASET_PATH), modality_configs=modality_config)
    traj = loader[EPISODE_ID]
    obs_modality_config = deepcopy(modality_config)
    obs_modality_config.pop("action", None)

    def parse_observation(obs: dict[str, Any]) -> dict[str, Any]:
        parsed = {"video": {}, "state": {}, "language": {}}
        for modality_name in ["video", "state", "language"]:
            for key in modality_config[modality_name].modality_keys:
                source_key = key if modality_name == "language" else f"{modality_name}.{key}"
                value = obs[source_key]
                parsed[modality_name][key] = [[value]] if isinstance(value, str) else value[None, :]
        return parsed

    for step in range(0, min(MAX_STEPS, len(traj)), ACTION_HORIZON):
        data_point = extract_step_data(traj, step, obs_modality_config, policy.embodiment_tag)
        obs = {f"state.{k}": v for k, v in data_point.states.items()}
        obs.update({f"video.{k}": np.asarray(v) for k, v in data_point.images.items()})
        for language_key in modality_config["language"].modality_keys:
            obs[language_key] = data_point.text
        action_chunk, info = policy.get_action(parse_observation(obs))
        for j in range(ACTION_HORIZON):
            t_index = step + j
            if t_index >= min(MAX_STEPS, len(traj)):
                break
            predicted = {k: np.asarray(v)[0, j] for k, v in action_chunk.items()}
            executions.append({
                "episode_id": EPISODE_ID,
                "step": t_index,
                "time": float(t_index * DT),
                "mode": "gr00t_policy",
                "task": data_point.text,
                "action": to_jsonable(predicted),
                "info": to_jsonable(info),
            })
else:
    for step, (_, row) in enumerate(episode_df.iterrows()):
        action_vec = flatten_row_arrays(row, action_columns)
        executions.append({
            "episode_id": EPISODE_ID,
            "step": step,
            "time": float(step * DT),
            "mode": "dataset_replay",
            "task": task_text,
            "action": {"vector": action_vec.tolist(), "columns": action_columns},
            "info": {},
        })

print(f"execution records: {len(executions)}")
executions[:2]

execution records: 64


[{'episode_id': 0,
  'step': 0,
  'time': 0.0,
  'mode': 'dataset_replay',
  'task': '',
  'action': {'vector': [0.0,
    0.0,
    0.4180138409137726,
    -0.1665557473897934,
    0.5165199637413025,
    3.0132288932800293,
    -0.2795029282569885,
    -0.2632471024990082,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    -0.22012978792190552,
    -0.4209160804748535,
    -0.13413986563682556,
    -2.3519082069396973,
    -0.19846679270267487,
    2.2142367362976074,
    0.0276581272482872,
    0.02907034009695053,
    0.000675040006171912,
    -0.02624109573662281,
    0.014119748026132584,
    -0.011163889430463314,
    -0.018826283514499664,
    0.006235593464225531,
    0.4180138409137726,
    -0.1665557473897934,
    0.5165199637413025,
    3.0132288932800293,
    -0.2795029282569885,
    -0.2632471024990082,
    0.0,
    0.4180138409137726,
    -0.1665557473897934,
    0.5165199637413025,
    -0.25011882185935974,
    -0.2758778929710388,
    -0.9280797243118286,
    0.96

## Save Executions And Trace Artifacts

Artifacts are written in three useful shapes: JSONL for rich execution records, CSV for quick analysis, and monitor JSON in the `{signal: [(time, value), ...]}` format expected by `src/monitors`.

In [ ]:
jsonl_path = OUTPUT_DIR / f"episode_{EPISODE_ID:06d}_executions.jsonl"
with jsonl_path.open("w") as f:
    for record in executions:
        f.write(json.dumps(record) + "\n")

rows = []
for record in executions:
    action = record["action"]
    vector = action.get("vector") if isinstance(action, dict) else None
    if vector is None and isinstance(action, dict):
        parts = [np.atleast_1d(np.asarray(v)).astype(float).ravel() for v in action.values()]
        vector = np.concatenate(parts).tolist() if parts else []
    row = {k: record[k] for k in ["episode_id", "step", "time", "mode", "task"]}
    for i, value in enumerate(vector or []):
        row[f"action_{i}"] = float(value)
    rows.append(row)

execution_table = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / f"episode_{EPISODE_ID:06d}_executions.csv"
execution_table.to_csv(csv_path, index=False)

print(jsonl_path)
print(csv_path)
execution_table.head()

In [ ]:
def finite_difference_norm(values: np.ndarray, dt: float) -> np.ndarray:
    if len(values) == 0:
        return values
    diffs = np.diff(values, axis=0, prepend=values[[0]])
    return np.linalg.norm(diffs, axis=1) / max(dt, 1e-9)

action_matrix = execution_table.filter(regex=r"^action_", axis=1).to_numpy(dtype=float)
times = execution_table["time"].to_numpy(dtype=float)
speed = finite_difference_norm(action_matrix, DT) if action_matrix.size else np.zeros(len(times))

# Placeholder environment signals. Replace these with simulator state extractors when running closed-loop rollouts.
dist_to_human = np.full(len(times), 1.0, dtype=float)
collision = np.zeros(len(times), dtype=float)
object_lifted = np.zeros(len(times), dtype=float)
if len(object_lifted):
    object_lifted[len(object_lifted) // 2 :] = 1.0

monitor_trace = {
    "ee_speed": list(zip(times.tolist(), speed.tolist())),
    "dist_to_human": list(zip(times.tolist(), dist_to_human.tolist())),
    "collision": list(zip(times.tolist(), collision.tolist())),
    "object_lifted": list(zip(times.tolist(), object_lifted.tolist())),
}

trace_path = OUTPUT_DIR / f"episode_{EPISODE_ID:06d}_monitor_trace.json"
with trace_path.open("w") as f:
    json.dump(monitor_trace, f, indent=2)

print(trace_path)
{name: series[:3] for name, series in monitor_trace.items()}

## Monitor STL Properties With RTAMT

This uses the repository's RTAMT wrapper. The Python package name is `rtamt`; install it in the notebook kernel environment if the import check fails.

In [ ]:
try:
    import rtamt  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Install the `rtamt` package in this notebook kernel to run STL monitoring.") from exc

from monitors import create_monitor
from monitors.report import format_spec_report
from monitors.specs import EVENTUALLY_OBJECT_LIFTED, NO_COLLISION, SAFE_DISTANCE, SPEED_LIMIT_NEAR_HUMAN
from monitors.trace import canonicalize_trace

trace = canonicalize_trace(monitor_trace)
specs = [NO_COLLISION, SAFE_DISTANCE, SPEED_LIMIT_NEAR_HUMAN, EVENTUALLY_OBJECT_LIFTED]

results = []
for spec in specs:
    monitor = create_monitor(spec["formula"], variables=spec["variables"], backend="rtamt")
    result = monitor.evaluate(trace)
    results.append({
        "name": spec["name"],
        "formula": spec["formula"],
        "satisfied": result.satisfied,
        "robustness": result.robustness,
        "time": result.time,
    })
    print(format_spec_report(spec["name"], result, trace))

results_df = pd.DataFrame(results)
results_path = OUTPUT_DIR / f"episode_{EPISODE_ID:06d}_stl_results.csv"
results_df.to_csv(results_path, index=False)
results_df

## Next Hooks

- Replace placeholder `dist_to_human`, `collision`, and `object_lifted` arrays with signals from a simulator state or real robot telemetry.
- For closed-loop simulation rollouts, log state at every step and keep the same monitor JSON contract.
- Add task-specific STL formulas next to `examples/toy_stl_rtamt_rulebook.yaml` once the signal names stabilize.